# 中关村学院-人工智能基础课 学生表现预测

By Luo Jien.

项目要求：根据给出特征，预测学生表现索引。

## 0x01 数据预处理

In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

df = pd.read_csv('student_performance.csv')

print(df.head())

   Hours Studied  Previous Scores Extracurricular Activities  Sleep Hours  \
0              7               99                        Yes            9   
1              4               82                         No            4   
2              8               51                        Yes            7   
3              5               52                        Yes            5   
4              7               75                         No            8   

   Sample Question Papers Practiced  Performance Index  
0                                 1               91.0  
1                                 2               65.0  
2                                 2               45.0  
3                                 2               36.0  
4                                 5               66.0  


上面的代码对数据集进行了读取和数据预处理。

## 0x02 数据集分割和可视化

In [3]:
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.feature_selection import SelectKBest, f_regression
ss = StandardScaler()
TOP_K = 3
# 使用SelectKBest进行特征选择
def select_k_best_features(df, target='Performance Index', k=TOP_K):
    """
    使用SelectKBest选择最佳的k个特征
    """
    # 选择数值型特征
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 移除目标列和ID列
    if target in numeric_columns:
        numeric_columns.remove(target)
    if 'Unnamed: 0' in numeric_columns:
        numeric_columns.remove('Unnamed: 0')
    
    X = df[numeric_columns]
    y = df[target]
    
    # 处理缺失值
    X = X.fillna(X.mean())
    
    # 使用SelectKBest选择特征
    selector = SelectKBest(score_func=f_regression, k=min(k, len(numeric_columns)))
    X_selected = selector.fit_transform(X, y)
    
    # 获取选中的特征名称
    selected_features = [numeric_columns[i] for i in selector.get_support(indices=True)]
    
    # 获取选中特征的得分
    selected_scores = selector.scores_[selector.get_support()]
    
    return selected_features, selected_scores
top_features, score = select_k_best_features(df)
X = df[top_features]
y = df['Performance Index']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Successfully loaded data, T size: ', len(X_train), '; V size: ', len(X_test), ';')

Successfully loaded data, T size:  8000 ; V size:  2000 ;


容易发现，Previous Scores 与 Performance Index 之间存在线性关系。

## 0x03 模型训练

In [6]:
import numpy as np
model = LinearRegression()
model.fit(X_train, y_train)

print(f"斜率：{model.intercept_:.4f}")
print(f"截距：{model.coef_[0]:.4f}")

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print("="*50)
print("模型评估指标")
print("="*50)
print("\n训练集:")
print(f"  MSE:  {mean_squared_error(y_train, y_pred_train):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
print(f"  MAE:  {mean_absolute_error(y_train, y_pred_train):.4f}")
print(f"  R²:   {r2_score(y_train, y_pred_train):.4f}")

print("\n测试集:")
print(f"  MSE:  {mean_squared_error(y_test, y_pred_test):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred_test):.4f}")
print(f"  R²:   {r2_score(y_test, y_pred_test):.4f}")

斜率：-32.7564
截距：2.8559
模型评估指标

训练集:
  MSE:  4.5631
  RMSE: 2.1361
  MAE:  1.6997
  R²:   0.9876

测试集:
  MSE:  4.5451
  RMSE: 2.1319
  MAE:  1.7044
  R²:   0.9877
